# 数据接入与图结构可视化

本 Notebook 演示：
1. 数据格式要求与加载过程
2. 图数据集构建过程（时间边 + 相关边）
3. 单日图结构可视化

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from graphrl.data.loader import load_real_data
from graphrl.data.dataset import create_enhanced_dataset
from graphrl.utils.seed import set_seed

set_seed(42)

## 1. 数据格式要求

`load_real_data` 要求输入文件包含以下列：
- `trade_date`：交易日期（datetime 或可转换字符串）
- `ts_code`：股票代码（字符串）
- `close`：收盘价
- 其余数值列均作为因子特征自动纳入

支持格式：`.pkl`、`.csv`、`.xlsx`

In [ ]:
DATA_PATH = '../CSI100_stock_data_with_factors.pkl'  # 修改为你的路径

price, bench, panel = load_real_data(DATA_PATH, max_assets=50, max_days=500)
    index_data_path='../CSI100_index.pkl',   # CSI100 指数基准

print(f'价格矩阵形状: {price.shape}（{price.shape[0]} 天 × {price.shape[1]} 支股票）')
print(f'面板数据列: {list(panel.columns)}')
print(f'\n价格矩阵前5行:')
price.iloc[:5, :5]

In [ ]:
# 基准净值曲线
plt.figure(figsize=(10, 3))
bench.plot()
plt.title('等权基准净值')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. 图数据集构建

In [ ]:
dataset, feat_names = create_enhanced_dataset(
    panel,
    seq_len=20,          # 用短窗口方便展示
    corr_window=10,
    corr_threshold=0.3,
)
print(f'图数据集样本数: {len(dataset)}')
print(f'特征列（共 {len(feat_names)} 个）: {feat_names[:8]}...')

In [ ]:
# 查看一个样本
sample = dataset[0]
print('节点特征 x:', sample.x.shape, '  (seq_len × n_assets, feat_dim)')
print('边索引 edge_index:', sample.edge_index.shape)
print('边属性 edge_attr:', sample.edge_attr.shape)
print('标签 y (下一日截面收益):', sample.y.shape)

# 区分时间边和相关边
n_assets = price.shape[1]
seq_len  = 20
last_step_offset = (seq_len - 1) * n_assets
is_corr_edge = (sample.edge_index[0] >= last_step_offset) & \
               (sample.edge_index[1] >= last_step_offset)
n_time = int((~is_corr_edge).sum())
n_corr = int(is_corr_edge.sum())
print(f'\n时间边数量: {n_time},  相关边数量: {n_corr}')

In [ ]:
# 相关系数分布
corr_weights = sample.edge_attr[is_corr_edge].numpy()
plt.figure(figsize=(6, 3))
plt.hist(corr_weights, bins=30, edgecolor='white')
plt.title(f'相关边权重分布（共 {n_corr} 条，阈值 0.3）')
plt.xlabel('Pearson 相关系数')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 收益预测目标分布
y_vals = sample.y.numpy()
plt.figure(figsize=(6, 3))
plt.hist(y_vals, bins=30, edgecolor='white')
plt.title('下一日截面收益分布')
plt.xlabel('日收益率')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'均值: {y_vals.mean():.4f},  标准差: {y_vals.std():.4f}')